As a quick project, I want to implement the paper [FR-Spec: Accelerating Large-Vocabulary Language Models via Frequency-Ranked Speculative Sampling}](https://aclanthology.org/2025.acl-long.198/) in vllm to make llm inference faster.

Here is the tldr:

When you do llm inference, some tokens are easier to predict than others. Do you really need your big, powerful, but slow model to predict these easy tokens? Can we use a smaller, faster model to predict these easy tokens? This is the idea behind speculative decoding: use a draft model, which is small and fast, to predict the easy tokens. Then use your big, slow, normal model called your target model for the hard tokens.This allows us to speedup llm inference.

However, the fr-spec authors noticed that 1/3 of the inference time is spent in the lm-head forward pass of the draft model. Since the lm-head is just a linear layer and multiples a matrix of dimension `hidden_dim x vocabulary`, the giant vocabulary becomes a bottleneck. So fr-spec proposes to prune the lm-head vocabulary. They found that we can throw away 75% of the least frequently used tokens in the vocabulary, shrinking the lm-head by 75%, and inference becomes 12% faster than eagle-2, the SOTA speculative decoding method at the time of writing.

The authors implemented this method in sglang, but not vllm. So today we want to implement this in vllm. Here in my PR.

I implemented the vocabulary pruning technique from fr-spec and added two new commands to the vllm api:
```sh
--draft-vocab-freq-path
```
gives a path to a file which states how frequently each token appears.
```sh
--draft-vocab-keep-threshold
```
tells you what percent of the most frequent tokens you should keep. If `--draft-vocab-keep-threshold 25`, then we keep the top 25% most frequently used tokens. This was pretty simple, as I could use the sglang implementation for reference.

As a sanity check, I evaluated speculative decoding for llama-3.1-8B-instruct using vanilla inference, eagle-2, and fr-spec on 100 prompts of mt-bench. The fr-spec used a similar setup and I even used the same pruned vocabulary that the fr-spec folks used.

My results made no sense:

Turns out there were several issues.
1. According to this PR, `cudagraphs` for draft models is currently broken in vllm. This means the target model gets a speedup from cudagraphs while the drafter does not. This makes it appear like the drafter produces a smaller speedup, giving an unfair advantage to vanilla. To fix, we disable all cuda/torch compilation `--compilation=None`.
2. Speculative decoding only works when we are memory bound, not compute bound. By default, vllm sets our `max-num-seq` to 256 and since our prompts are so short, we become compute bound, not memory bound. To fix this, we set `--max-num-seq 1`. The fr-spec authors don't mention this in their paper, but they do the same thing here. A notable footgun is that `max-num-seq` controls the number of user prompts processed at a time, not the actual batch size. In fact given a speculative decoding tree of depth `d` and branching factor `b`, the drafter batch size is `b^d` with `d` forward passes and the target batch size is `b^0 + b^1 + ... + b^d` with 1 forward pass
3. Since we are memory bound, our drafter should be able to guess multiple tokens without increasing the time, thereby fully using all of the compute available to us. So instead of just using 1 speculative token, we can guess 5 different tokens. However, we have to be careful how we do this. We want to increase the batch size of each drafter, rather than the number of drafter forward passes, and therefore we should increase the branching factor of the speculative tree rather than the depth of the speculative decoding tree. So we set `--spec-token-tree-depth 3` and `--spec-token-tree-branching 3` for a total of 3+9+27=39 speculative tokens. The fr-spec authors use 60 speculative tokens arranged in a tree. A notable footgun is that you specify the speculative tree differently in vllm and sglang.
4. The kv-caching is broken for drafter models in vllm. Let's say the drafter suggests 3 speculative tokens for the next 3 tokens in a sentence. Because it stores the previous keys and values in the kv-cache, the drafter forward passe
Then the verifier 

Rerunning these results, we got